# CinderTrace — cached Sri Lanka pixel and pass audit

This continues your completed observation-evidence pilot. **No Earthdata login or new science downloads are needed.** It reads the existing `science_cache`, verifies all 68 files against the returned receipts, checks individual fire pixels and viewing geometry, and compares one-to-one satellite passes.

The verified pilot metadata and 335 corresponding standard archive observations are embedded. No input ZIP upload is needed. Keep your completed pilot notebook, original results and cache. This notebook writes a separate output folder and downloads one review ZIP. It does not change the frontend, restore missing months, fill gaps or claim complete observation coverage.

## 1. Install the readers
This may download Python packages, but the audit makes no NASA data requests. In an existing Colab kernel most dependencies may already be installed.

In [102]:
%pip -q install numpy pandas pyarrow shapely h3 scipy netCDF4


## 2. Locate your existing cache
In Colab, the default uses the exact Drive folder from your completed pilot. It may ask for Drive access, not Earthdata login. If you moved the folder, update `PILOT_ROOT`. In VS Code, point it at the local folder containing `science_cache`. Do not point it at the small results ZIP: that ZIP intentionally excludes the science files.

In [1]:
from pathlib import Path
import sys,json,hashlib,base64,zlib,zipfile
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PILOT_ROOT=Path('/content/drive/MyDrive/CinderTrace_Observation_Evidence_v1')
else:
    PILOT_ROOT=Path('data/atlas/observation_evidence_run_v1')
CACHE=PILOT_ROOT/'science_cache'
assert CACHE.is_dir(),'Set PILOT_ROOT to the original folder containing science_cache. No download was attempted.'
OUTPUT_ROOT=PILOT_ROOT/'pixel_pass_audit_v1'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RESULTS=OUTPUT_ROOT/'results';RESULTS.mkdir(exist_ok=True)
print('Reading cache:',CACHE.resolve())
print('Writing new results:',RESULTS.resolve())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading cache: /content/drive/MyDrive/CinderTrace_Observation_Evidence_v1/science_cache
Writing new results: /content/drive/MyDrive/CinderTrace_Observation_Evidence_v1/pixel_pass_audit_v1/results


## 3. Verify the embedded inputs
These come from your checked results ZIP and the frozen standard-data checkpoint. Type labels are carried from that archive only after a unique coordinate/time match; they are not independent confirmation of vegetation burning.

In [2]:
INPUT_BYTES=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(INPUT_BYTES).hexdigest()=='7369e95ba955d0abc81e19781884462eee4b2fdaed56ad05252368e1ae1e7737'
INPUTS=json.loads(INPUT_BYTES)
import pandas as pd
ARCHIVE=pd.DataFrame(INPUTS['standard_observations'])
ARCHIVE['acq_datetime']=pd.to_datetime(ARCHIVE.acq_datetime,utc=True)
PREVIOUS=pd.DataFrame(INPUTS['previous_granules'])
assert len(ARCHIVE)==335 and len(PREVIOUS)==34
(OUTPUT_ROOT/'verified_inputs.json').write_bytes(INPUT_BYTES)
print('Frozen inputs:',len(PREVIOUS),'fire granules;',len(ARCHIVE),'native standard observations.')


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Frozen inputs: 34 fire granules; 335 native standard observations.


## 4. Load the method
Every local native fire pixel is checked against the fire mask and sparse arrays. Pixel quality, confidence, residual bowtie status and solar/viewing angles are retained. Archived type attribution uses a unique same-sensor coordinate match within two metres and the granule's acquisition interval; ambiguous and unmatched cases remain explicit.

Adjacent granules are grouped by sensor/date/day-night with at most a six-minute gap. Pass pairing uses each pass at most once within each 30/60/90-minute sensitivity. Mutual nearest clear pixel centres are compared at 250/500 metres with all-known, ≤40° and ≤60° viewing screens. **These are centre-proximity diagnostics, not identical physical footprints or complete area coverage.** The same data appears in several sensitivities; those are not independent samples. Granule times are not exact pixel observation times.

In [3]:
import types
BASE_METHOD_SOURCE='"""Small VIIRS standard-data observation evidence pilot; never fills archive zeros.\nPublic metadata discovery runs without credentials. Science screening requires\nEarthdata-authorized files and is a pixel-centre diagnostic, not area completeness.\n"""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,json,re,time,urllib.request,urllib.parse,urllib.error\nimport numpy as np\nimport pandas as pd\n\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/\'data/atlas/observation_evidence_pilot_v1\'\nPRODUCTS={\n \'VNP14IMG\':{\'collection\':\'C2734202914-LPCLOUD\',\'version\':\'002\',\'satellite\':\'Suomi-NPP\',\'kind\':\'fire\'},\n \'VJ114IMG\':{\'collection\':\'C2734197957-LPCLOUD\',\'version\':\'002\',\'satellite\':\'NOAA-20\',\'kind\':\'fire\'},\n \'VNP03IMG\':{\'collection\':\'C2105092163-LAADS\',\'version\':\'2\',\'satellite\':\'Suomi-NPP\',\'kind\':\'geolocation\'},\n \'VJ103IMG\':{\'collection\':\'C2105086226-LAADS\',\'version\':\'2.1\',\'satellite\':\'NOAA-20\',\'kind\':\'geolocation\'}}\nDAYS={\'2021-08-15\':\'recorded_zero_control_not_no_fire_truth\',\n \'2021-08-19\':\'positive_control\', \'2022-08-04\':\'listed_SNPP_missing_standard_date\',\n \'2022-08-12\':\'post_interruption_recovery_candidate\',\n \'2024-05-31\':\'listed_SNPP_missing_standard_date\',\n \'2024-06-05\':\'post_interruption_recovery_candidate\'}\nBOUNDARY_SHA=\'838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e\'\nSOURCES={\n \'fire_guide\':\'https://ladsweb.modaps.eosdis.nasa.gov/archive/Document%20Archive/Science%20Data%20Product%20Documentation/VIIRS_C2_AF-375m_User_Guide_1.0.pdf\',\n \'snpp_geolocation\':\'https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VNP03IMG/\',\n \'noaa20_geolocation\':\'https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VJ103IMG\',\n \'cmr\':\'https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html\'}\nPOLICY={\'pilot_id\':\'lk_viirs_observation_evidence_v1\',\'days\':DAYS,\'products\':PRODUCTS,\'sources\':SOURCES,\n \'sampling\':\'purposive schema/outage feasibility sample, not independent model validation\',\n \'standard_only\':True,\'h3_resolution\':8,\'candidate_time_windows_minutes\':[30,60,90],\n \'candidate_pairing\':\'all cross-sensor granule candidates within time windows; not assumed independent or one-to-one\',\n \'support\':\'pixel centres only; no footprint areas, cloud-free area fractions or complete coverage inferred\',\n \'screen\':\'fire-mask classes 5/7/8/9, nominal input QA bits 0–6, residual bowtie bit 22 absent; not an official replacement algorithm\',\n \'scope\':\'all native fire-mask classes, not vegetation-only model target; type attribution not independently available\',\n \'zero_rule\':\'missing catalogue, missing companion, failed download or screen is unavailable, never an observed zero\',\n \'replacement_enabled\':False,\'early_history_bands_enabled\':False,\'coverage_verified\':False,\'frontend_changed\':False}\n\ndef sha(raw):return hashlib.sha256(raw).hexdigest()\ndef acquisition(name):\n    match=re.search(r\'\\.A(\\d{7})\\.(\\d{4})\\.\',name)\n    if not match:raise ValueError(\'Unrecognized acquisition filename\')\n    return \'.\'.join(match.groups())\n\ndef query(params,destination):\n    url=\'https://cmr.earthdata.nasa.gov/search/granules.json?\'+urllib.parse.urlencode(params)\n    receipt=destination.with_suffix(\'.receipt.json\')\n    if destination.exists() and receipt.exists():\n        raw=destination.read_bytes();r=json.loads(receipt.read_text())\n        if r[\'url\']!=url or r[\'sha256\']!=sha(raw):raise ValueError(\'Changed metadata cache\')\n        return json.loads(raw),r\n    for attempt in range(4):\n        try:\n            with urllib.request.urlopen(urllib.request.Request(url,headers={\'User-Agent\':\'CinderTrace-observation-evidence/1.0\'}),timeout=60) as response:\n                raw=response.read();hits=response.headers.get(\'CMR-Hits\')\n            body=json.loads(raw)\n            if not isinstance(body.get(\'feed\',{}).get(\'entry\'),list):raise ValueError(\'Unexpected CMR schema\')\n            r={\'url\':url,\'sha256\':sha(raw),\'bytes\':len(raw),\'cmr_hits\':int(hits) if hits is not None else None,\'retrieved_utc\':datetime.now(timezone.utc).isoformat()}\n            destination.write_bytes(raw);receipt.write_text(json.dumps(r,indent=2)+\'\\n\')\n            return body,r\n        except (urllib.error.URLError,TimeoutError):\n            if attempt==3:raise\n            time.sleep(min(2**attempt,4))\n\ndef normalize(entry,product,day):\n    spec=PRODUCTS[product];name=entry[\'producer_granule_id\']\n    if entry.get(\'collection_concept_id\')!=spec[\'collection\'] or not name.startswith(product+\'.\'):raise ValueError(\'Wrong collection/product\')\n    urls=[l[\'href\'] for l in entry.get(\'links\',[]) if not l.get(\'inherited\') and l.get(\'rel\',\'\').endswith(\'/data#\') and l.get(\'href\',\'\').startswith(\'https://\') and urllib.parse.urlsplit(l[\'href\']).path.endswith(\'.nc\')]\n    if len(urls)!=1:raise ValueError(\'Expected one science NetCDF link\')\n    return {\'id\':entry[\'id\'],\'product\':product,\'satellite\':spec[\'satellite\'],\'kind\':spec[\'kind\'],\n     \'name\':name,\'acquisition\':acquisition(name),\'date\':day,\'time_start\':entry[\'time_start\'],\'time_end\':entry[\'time_end\'],\n     \'day_night\':entry.get(\'day_night_flag\',\'UNKNOWN\'),\'download_url\':urls[0],\n     \'catalogue_size_mb\':float(entry.get(\'granule_size\',0)),\'metadata_entry_sha256\':sha(json.dumps(entry,sort_keys=True).encode()),\n     \'catalogue_footprint_only\':True}\n\ndef join_companions(records):\n    fire=[r for r in records if r[\'kind\']==\'fire\'];result=[]\n    for f in fire:\n        g=[g for g in records if g[\'kind\']==\'geolocation\' and g[\'satellite\']==f[\'satellite\'] and g[\'acquisition\']==f[\'acquisition\']]\n        status=\'unique_catalogue_companion\' if len(g)==1 else \'missing_companion\' if not g else \'ambiguous_companion\'\n        if len(g)==1 and (pd.Timestamp(g[0][\'time_start\'])!=pd.Timestamp(f[\'time_start\']) or pd.Timestamp(g[0][\'time_end\'])!=pd.Timestamp(f[\'time_end\'])):status=\'time_extent_mismatch\'\n        result.append({\'fire_id\':f[\'id\'],\'geo_id\':g[0][\'id\'] if len(g)==1 else None,\'satellite\':f[\'satellite\'],\'date\':f[\'date\'],\'acquisition\':f[\'acquisition\'],\'status\':status})\n    return result\n\ndef time_candidates(records):\n    fire=[r for r in records if r[\'kind\']==\'fire\'];rows=[]\n    for a in fire:\n        if a[\'satellite\']!=\'Suomi-NPP\':continue\n        for b in fire:\n            if b[\'satellite\']!=\'NOAA-20\' or a[\'date\']!=b[\'date\']:continue\n            # Use separation of granule midpoints. Date-boundary candidates are out of this small pilot.\n            am=pd.Timestamp(a[\'time_start\'])+(pd.Timestamp(a[\'time_end\'])-pd.Timestamp(a[\'time_start\']))/2\n            bm=pd.Timestamp(b[\'time_start\'])+(pd.Timestamp(b[\'time_end\'])-pd.Timestamp(b[\'time_start\']))/2\n            dt=abs((am-bm).total_seconds())/60\n            if dt<=90:rows.append({\'snpp_id\':a[\'id\'],\'noaa20_id\':b[\'id\'],\'date\':a[\'date\'],\'midpoint_separation_minutes\':dt,\n             \'within_30\':dt<=30,\'within_60\':dt<=60,\'within_90\':True,\'day_night_agrees\':a[\'day_night\']==b[\'day_night\'] and a[\'day_night\'] in [\'DAY\',\'NIGHT\'],\n             \'status\':\'time_candidate_only_pixel_overlap_unverified\'})\n    return rows\n\ndef discover(boundary_path=None,output=None):\n    out=Path(OUT if output is None else output);out.mkdir(parents=True,exist_ok=True);rawdir=out/\'cmr\';rawdir.mkdir(exist_ok=True)\n    bp=Path(boundary_path or ROOT/\'data/atlas/canonical_lk_2000_2024_v1/region.geojson\');raw=bp.read_bytes()\n    if sha(raw)!=BOUNDARY_SHA:raise ValueError(\'Boundary differs from verified study region\')\n    from shapely.geometry import shape\n    region=shape(json.loads(raw)[\'features\'][0][\'geometry\']);bbox=\',\'.join(map(str,region.bounds));(out/\'region.geojson\').write_bytes(raw)\n    records=[];counts=[]\n    for day in DAYS:\n        for product,spec in PRODUCTS.items():\n            found=[];page=1;expected=None\n            while True:\n                params={\'collection_concept_id\':spec[\'collection\'],\'bounding_box\':bbox,\'temporal\':day+\'T00:00:00Z,\'+day+\'T23:59:59.999Z\',\'page_size\':200,\'page_num\':page}\n                body,receipt=query(params,rawdir/f\'{product}_{day}_{page}.json\');entries=body[\'feed\'][\'entry\']\n                if expected is None:expected=receipt[\'cmr_hits\']\n                elif receipt[\'cmr_hits\']!=expected:raise ValueError(\'Catalogue changed during pagination; retry in a fresh output folder\')\n                found.extend(entries)\n                if len(entries)<200:break\n                page+=1\n                if page>100:raise ValueError(\'Unexpected pilot catalogue size\')\n            if expected is not None and len(found)!=expected:raise ValueError(\'Incomplete CMR pagination\')\n            if len({e[\'id\'] for e in found})!=len(found):raise ValueError(\'Duplicate CMR entries\')\n            # Temporal queries overlap intervals; retain granules whose start falls on the requested UTC day.\n            found=[e for e in found if e[\'time_start\'][:10]==day]\n            normalized=[normalize(e,product,day) for e in found];records.extend(normalized)\n            counts.append({\'date\':day,\'role\':DAYS[day],\'product\':product,\'records\':len(normalized),\'status\':\'catalogue_listed\' if normalized else \'no_catalogue_records_not_zero_observations\'})\n        print(\'Catalogued\',day,flush=True)\n    joins=join_companions(records);candidates=time_candidates(records)\n    report={\'policy\':POLICY,\'created_utc\':datetime.now(timezone.utc).isoformat(),\'boundary_sha256\':sha(raw),\'bbox\':list(region.bounds),\n     \'records\':records,\'daily_catalogue\':counts,\'companions\':joins,\'time_candidates\':candidates,\n     \'catalogue_size_mb\':sum(r[\'catalogue_size_mb\'] for r in records),\n     \'status\':\'metadata_only_science_screen_pending\',\'science_files_verified\':0,\'usable_coverage_verified\':False}\n    (out/\'inventory.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\');(out/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    return report\n\nclass ScienceAccessRequired(RuntimeError):pass\n\ndef download(record,session,cache):\n    """Resumable per file; stores no auth headers, cookies, tokens or passwords."""\n    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)\n    filename=Path(urllib.parse.urlsplit(record[\'download_url\']).path).name\n    if filename!=record[\'name\']+\'.nc\' and filename!=record[\'name\']:raise ValueError(\'URL/name mismatch\')\n    target=cache/filename;rp=cache/(filename+\'.receipt.json\')\n    if target.exists() and rp.exists():\n        r=json.loads(rp.read_text());raw=target.read_bytes()\n        if r[\'id\']!=record[\'id\'] or r[\'sha256\']!=sha(raw) or r[\'bytes\']!=len(raw):raise ValueError(\'Changed verified science cache\')\n        return target,r\n    part=cache/(filename+\'.part\');reason=\'network\'\n    for attempt in range(4):\n        try:\n            with session.get(record[\'download_url\'],stream=True,timeout=(30,180)) as response:\n                if response.status_code!=200:\n                    reason=\'HTTP \'+str(response.status_code)\n                    if response.status_code in (401,403):break\n                    response.raise_for_status()\n                expected=int(response.headers.get(\'Content-Length\',0));digest=hashlib.sha256();n=0\n                with part.open(\'wb\') as handle:\n                    for chunk in response.iter_content(1024*1024):\n                        if chunk:handle.write(chunk);digest.update(chunk);n+=len(chunk)\n                if expected and n!=expected:raise OSError(\'Incomplete content length\')\n            with part.open(\'rb\') as f:signature=f.read(8)\n            if signature!=b\'\\x89HDF\\r\\n\\x1a\\n\':raise ValueError(\'Response is not NetCDF4/HDF5 science data\')\n            r={\'id\':record[\'id\'],\'product\':record[\'product\'],\'acquisition\':record[\'acquisition\'],\'bytes\':n,\'sha256\':digest.hexdigest(),\n             \'retrieved_utc\':datetime.now(timezone.utc).isoformat(),\'http_status\':200,\'status\':\'content_length_and_signature_verified_not_provider_checksum\'}\n            part.replace(target);rp.write_text(json.dumps(r,indent=2)+\'\\n\');return target,r\n        except Exception as error:\n            # Do not print request exceptions; they can contain provider credential redirects.\n            reason=type(error).__name__\n            if attempt<3:time.sleep(min(2**attempt,4))\n        finally:\n            if part.exists():part.unlink()\n    if reason in [\'HTTP 401\',\'HTTP 403\']:raise ScienceAccessRequired(f\'{filename}: {reason}. Check Earthdata authorization for this provider; no zero written.\')\n    raise RuntimeError(f\'{filename}: {reason}. No zero written. Verified prior files retained; review Earthdata/provider access.\') from None\n\ndef variables(dataset,prefix=\'\'):\n    result={prefix+name:var for name,var in dataset.variables.items()}\n    for name,group in dataset.groups.items():result.update(variables(group,prefix+name+\'/\'))\n    return result\n\ndef find_variable(dataset,name):\n    norm=lambda s:re.sub(\'[^a-z0-9]\',\'\',s.lower())\n    hits=[(path,var) for path,var in variables(dataset).items() if norm(path.split(\'/\')[-1])==norm(name)]\n    if len(hits)!=1:raise ValueError(\'Missing or ambiguous science variable: \'+name)\n    return hits[0]\n\ndef screen_arrays(mask,qa,latitude,longitude,region):\n    """Chunk kernel: preserve every local class; good centres are an explicit screen."""\n    from shapely import intersects_xy\n    m=np.ma.asarray(mask);q=np.ma.asarray(qa);lat=np.ma.asarray(latitude);lon=np.ma.asarray(longitude)\n    if len(m.shape)!=2 or len({m.shape,q.shape,lat.shape,lon.shape})!=1:raise ValueError(\'Fire/geolocation grid mismatch\')\n    lat=np.ma.filled(lat.astype(float),np.nan);lon=np.ma.filled(lon.astype(float),np.nan)\n    # Invalid/masked QA must never be converted to nominal (zero).\n    goodgeo=np.isfinite(lat)&np.isfinite(lon)&(abs(lat)<=90)&(abs(lon)<=180)\n    local=goodgeo&intersects_xy(region,lon,lat)\n    mv=np.ma.filled(m.astype(float),np.nan);qv=np.ma.filled(q.astype(np.uint32),np.uint32(2**32-1))\n    known=~np.ma.getmaskarray(m)&np.isfinite(mv)&np.isin(mv,np.arange(10))\n    if (local&~known).any():raise ValueError(\'Unknown or missing local fire-mask class\')\n    nominal=(qv&127)==0;not_duplicate=(qv&(1<<22))==0\n    clear=local&np.isin(mv,[5,7,8,9])&nominal&not_duplicate&~np.ma.getmaskarray(q)\n    fires=clear&np.isin(mv,[7,8,9])\n    return {\'local\':local,\'mask\':mv,\'qa\':qv,\'qa_valid\':~np.ma.getmaskarray(q),\'clear\':clear,\'fires\':fires,\'latitude\':lat,\'longitude\':lon}\n\ndef screen_file(fire_path,geo_path,record,region):\n    from netCDF4 import Dataset\n    import h3\n    rows=[];summary={\'fire_id\':record[\'id\'],\'product\':record[\'product\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\n      \'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\'day_night\':record[\'day_night\'],\n      \'local_geolocated_centres\':0,\'nominal_clear_centres\':0,\'screened_fire_centres\':0,\'local_residual_bowtie_centres\':0,\n      \'scope\':\'all native fire classes; no vegetation type attribution\',\'status\':\'pixel_centre_screen_only_not_complete_coverage\'}\n    with Dataset(fire_path) as fire,Dataset(geo_path) as geo:\n        fm,mask=find_variable(fire,\'fire mask\');fq,qa=find_variable(fire,\'algorithm QA\');gl,lat=find_variable(geo,\'latitude\');go,lon=find_variable(geo,\'longitude\')\n        if len(mask.shape)!=2 or mask.shape!=lat.shape or mask.shape!=lon.shape or mask.shape!=qa.shape:raise ValueError(\'Companion grid shape mismatch\')\n        summary.update({\'fire_mask_variable\':fm,\'qa_variable\':fq,\'latitude_variable\':gl,\'longitude_variable\':go,\'shape\':list(mask.shape)})\n        # Independent sparse-fire coordinates check the zero-based line/sample alignment.\n        _,pl=find_variable(fire,\'FP_line\');_,ps=find_variable(fire,\'FP_sample\');_,plat=find_variable(fire,\'FP_latitude\');_,plon=find_variable(fire,\'FP_longitude\')\n        alignment=[]\n        for r,c,a,b in zip(pl[:100],ps[:100],plat[:100],plon[:100]):\n            r=int(r);c=int(c)\n            if r<0 or c<0 or r>=mask.shape[0] or c>=mask.shape[1]:raise ValueError(\'Invalid zero-based sparse-fire grid index\')\n            x=float(lat[r,c]);y=float(lon[r,c]);delta=max(abs(x-float(a)),abs(((y-float(b)+180)%360)-180))\n            if not np.isfinite(delta) or delta>.01:raise ValueError(\'Sparse-fire geolocation does not align with companion grid\')\n            alignment.append(delta)\n        summary[\'alignment_check_pixels\']=len(alignment);summary[\'alignment_max_error_degrees\']=max(alignment) if alignment else None\n        summary[\'alignment_status\']=\'sparse_fire_coordinates_agree\' if alignment else \'no_sparse_fires_only_catalogue_time_and_shape_checks\'\n        for start in range(0,mask.shape[0],256):\n            sl=slice(start,min(start+256,mask.shape[0]));x=screen_arrays(mask[sl,:],qa[sl,:],lat[sl,:],lon[sl,:],region)\n            summary[\'local_geolocated_centres\']+=int(x[\'local\'].sum());summary[\'nominal_clear_centres\']+=int(x[\'clear\'].sum());summary[\'screened_fire_centres\']+=int(x[\'fires\'].sum())\n            summary[\'local_residual_bowtie_centres\']+=int((x[\'local\']&x[\'qa_valid\']&((x[\'qa\']&(1<<22))!=0)).sum())\n            for cls in range(10):summary[\'class_\'+str(cls)]=summary.get(\'class_\'+str(cls),0)+int((x[\'local\']&(x[\'mask\']==cls)).sum())\n            ix=x[\'local\'];la=x[\'latitude\'][ix];lo=x[\'longitude\'][ix]\n            if not len(la):continue\n            chunk=pd.DataFrame({\'cell\':[h3.latlng_to_cell(float(a),float(b),8) for a,b in zip(la,lo)],\n              \'local_centres\':1,\'clear_centres\':x[\'clear\'][ix].astype(int),\'fire_centres\':x[\'fires\'][ix].astype(int)})\n            rows.append(chunk.groupby(\'cell\',as_index=False).sum())\n    cells=pd.concat(rows).groupby(\'cell\',as_index=False).sum() if rows else pd.DataFrame(columns=[\'cell\',\'local_centres\',\'clear_centres\',\'fire_centres\'])\n    for name in [\'fire_id\',\'satellite\',\'date\',\'time_start\',\'time_end\']:cells[name]=summary[name]\n    # Presence of a centre does not imply coverage of the full H3 cell.\n    return summary,cells\n\ndef candidate_diagnostics(candidates,cells):\n    rows=[]\n    for c in candidates:\n        base=dict(c);a=cells[cells.fire_id.eq(c[\'snpp_id\'])];b=cells[cells.fire_id.eq(c[\'noaa20_id\'])]\n        if not len(a) or not len(b):\n            base.update({\'status\':\'one_or_both_screens_missing_or_no_regional_centres\',\'common_clear_centre_cells\':None,\n              \'snpp_active_cells_on_common_support\':None,\'noaa20_active_cells_on_common_support\':None});rows.append(base);continue\n        # Cell labels approximate spatial co-location; actual pixel footprints differ.\n        common=a[a.clear_centres.gt(0)].merge(b[b.clear_centres.gt(0)],on=\'cell\',suffixes=(\'_snpp\',\'_noaa20\'))\n        base.update({\'status\':\'coarse_cell_centre_overlap_diagnostic_not_matched_pixel_footprints\',\n         \'common_clear_centre_cells\':len(common),\n         \'snpp_active_cells_on_common_support\':int(common.fire_centres_snpp.gt(0).sum()) if len(common) else None,\n         \'noaa20_active_cells_on_common_support\':int(common.fire_centres_noaa20.gt(0).sum()) if len(common) else None})\n        rows.append(base)\n    return pd.DataFrame(rows)\n\ndef daily_screen_status(inventory,summaries):\n    rows=[]\n    for day in DAYS:\n        for satellite in [\'Suomi-NPP\',\'NOAA-20\']:\n            planned=[r for r in inventory[\'records\'] if r[\'kind\']==\'fire\' and r[\'date\']==day and r[\'satellite\']==satellite]\n            done=[r for r in summaries if r[\'date\']==day and r[\'satellite\']==satellite]\n            all_done=bool(planned) and len(done)==len(planned)\n            local=sum(r[\'local_geolocated_centres\'] for r in done)\n            supported=all_done and local>0\n            rows.append({\'date\':day,\'satellite\':satellite,\'catalogue_fire_granules\':len(planned),\'screened_granules\':len(done),\n                \'status\':\'catalogue_absent_not_zero_observations\' if not planned else \'incomplete_science_screen\' if not all_done else \'no_local_centres_not_zero_activity\' if not local else \'catalogued_granules_screened_coverage_unverified\',\n                \'screened_fire_centres\':sum(r[\'screened_fire_centres\'] for r in done) if supported else None,\n                \'nominal_clear_centres\':sum(r[\'nominal_clear_centres\'] for r in done) if supported else None,\n                \'observing_coverage_verified\':False})\n    return pd.DataFrame(rows)\n\ndef run_science(inventory,session,cache,output):\n    from shapely.geometry import shape\n    out=Path(output);out.mkdir(parents=True,exist_ok=True);source=Path(inventory);inv=json.loads(source.read_text());region_raw=(source.parent/\'region.geojson\').read_bytes()\n    if sha(region_raw)!=BOUNDARY_SHA:raise ValueError(\'Boundary changed\')\n    region=shape(json.loads(region_raw)[\'features\'][0][\'geometry\']);lookup={r[\'id\']:r for r in inv[\'records\']};summaries=[];cells=[];receipts=[];failures=[]\n    # Each completed swath is retained immediately, so later access/schema failures do not lose prior work.\n    access_stopped=False\n    for join in inv[\'companions\']:\n        if access_stopped:\n            failures.append({**join,\'status\':\'not_attempted_after_access_denial\'});continue\n        if join[\'status\']!=\'unique_catalogue_companion\':failures.append({**join,\'status\':\'unavailable_companion\'});continue\n        f=lookup[join[\'fire_id\']];g=lookup[join[\'geo_id\']]\n        try:\n            fp,fr=download(f,session,cache);gp,gr=download(g,session,cache);receipts.extend([fr,gr])\n            summary,cell=screen_file(fp,gp,f,region);summaries.append(summary);cells.append(cell)\n            pd.DataFrame(summaries).to_parquet(out/\'granule_screening.parquet\',index=False)\n            pd.concat(cells,ignore_index=True).to_parquet(out/\'cell_centre_support.parquet\',index=False)\n            print(\'Screened\',f[\'name\'],flush=True)\n        except Exception as error:\n            # Error text may include authenticated URLs from a third-party library; retain safe type only.\n            failures.append({**join,\'status\':\'unavailable_science_screen\',\'error_type\':type(error).__name__})\n            if isinstance(error,ScienceAccessRequired):access_stopped=True\n            print(\'Unavailable screen:\',f[\'name\'],type(error).__name__,flush=True)\n        finally:\n            (out/\'download_receipts.json\').write_text(json.dumps(receipts,indent=2)+\'\\n\');(out/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    combined=pd.concat(cells,ignore_index=True) if cells else pd.DataFrame(columns=[\'fire_id\',\'cell\',\'clear_centres\',\'fire_centres\'])\n    pd.DataFrame(summaries,columns=None if summaries else [\'fire_id\',\'satellite\',\'date\',\'status\']).to_parquet(out/\'granule_screening.parquet\',index=False)\n    combined.to_parquet(out/\'cell_centre_support.parquet\',index=False)\n    (out/\'download_receipts.json\').write_text(json.dumps(receipts,indent=2)+\'\\n\')\n    (out/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    daily=daily_screen_status(inv,summaries)\n    daily.to_parquet(out/\'daily_screen_status.parquet\',index=False)\n    diagnostic=candidate_diagnostics(inv[\'time_candidates\'],combined);diagnostic.to_parquet(out/\'candidate_pair_diagnostics.parquet\',index=False)\n    decision={\'pilot_id\':POLICY[\'pilot_id\'],\'inventory_sha256\':sha(source.read_bytes()),\'planned_fire_granules\':len(inv[\'companions\']),\n     \'screened_fire_granules\':len(summaries),\'failed_or_unavailable\':len(failures),\'access_denial_stopped_downloads\':access_stopped,\'status\':\'exploratory_pixel_centre_evidence_only\',\n     \'zero_months_restored\':0,\'replacement_validated\':False,\'gap_fill_enabled\':False,\'coverage_verified\':False,\n     \'independent_uncertainty_validation_completed\':False,\'frontend_changed\':False,\n     \'limitations\':[\'Small purposive sample; no model tuning or validation scores\',\'Granule times, not exact per-pixel observation times\',\n       \'Coarse cell-centre overlap is not matched physical footprints or full regional area support\',\'Candidate pairs may share granules; do not sum pair counts\',\n       \'No vegetation type attribution: do not compare directly to vegetation-only transfer scores\',\'Missing files and failed screens remain unavailable\']}\n    (out/\'decision.json\').write_text(json.dumps(decision,indent=2)+\'\\n\');(out/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    decision[\'files\']={}\n    for path in sorted(out.iterdir()):\n        if path.is_file() and path.name not in [\'manifest.json\']:\n            raw=path.read_bytes();decision[\'files\'][path.name]={\'bytes\':len(raw),\'sha256\':sha(raw)}\n    (out/\'manifest.json\').write_text(json.dumps(decision,indent=2,allow_nan=False)+\'\\n\')\n    return decision\n\n'
AUDIT_METHOD_SOURCE='"""Reprocess verified pilot cache without network or credentials.\nNearest pixel centres are diagnostics, not physical footprint equivalence.\n"""\nfrom pathlib import Path\nimport hashlib,json,urllib.parse\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial import cKDTree\nfrom scipy.optimize import linear_sum_assignment\nfrom . import atlas_observation_evidence as e\n\nPOLICY={\'audit_id\':\'lk_cached_pixel_pass_audit_v1\',\'network_downloads\':False,\n \'pass_grouping\':\'same satellite/UTC date/catalogue day-night; successive intervals separated by at most six minutes; pilot edges may truncate passes\',\n \'pairing\':\'one-to-one maximum-cardinality minimum-midpoint-time assignment separately at 30/60/90 minutes; unknown/mixed day-night excluded\',\n \'matching\':\'mutual nearest nominal-clear pixel centres on sphere; 250/500 metre sensitivity; not footprint intersection or simultaneity\',\n \'view_screens_degrees\':[None,40,60],\'archive_match_metres\':2,\n \'archive_match\':\'same satellite and granule interval; mutual unique coordinate candidate within two metres; preserve ambiguous/unmatched states\',\n \'archive_type\':\'carried from uniquely matched standard observation, not independently validated burning type\',\n \'coverage_verified\':False,\'replacement_validated\':False,\'uncertainty_validated\':False,\'frontend_changed\':False}\nRADIUS=6371008.8\n\ndef require(ok,message):\n    if not ok:raise ValueError(message)\ndef file_sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\ndef xyz(lat,lon):\n    a=np.radians(np.asarray(lat,float));b=np.radians(np.asarray(lon,float))\n    return np.column_stack((np.cos(a)*np.cos(b),np.cos(a)*np.sin(b),np.sin(a)))\ndef chord(metres):return 2*np.sin(metres/(2*RADIUS))\ndef mutual_neighbours(a,b):\n    if not len(a) or not len(b):return np.array([],int),np.array([],int),np.array([],float)\n    aa=xyz(a.latitude,a.longitude);bb=xyz(b.latitude,b.longitude)\n    da,ia=cKDTree(bb).query(aa);_,ib=cKDTree(aa).query(bb)\n    ix=np.flatnonzero(ib[ia]==np.arange(len(aa)));iy=ia[ix]\n    return ix,iy,2*RADIUS*np.arcsin(np.minimum(1,da[ix]/2))\n\ndef archive_match(native,archive):\n    out=native.copy();out[\'archive_match_status\']=\'unmatched\';out[\'archive_observation_id\']=None;out[\'archive_type\']=pd.Series(pd.NA,index=out.index,dtype=\'Int64\');out[\'archive_distance_metres\']=np.nan\n    proposals={};reverse={}\n    for i,row in out.iterrows():\n        start=pd.Timestamp(row.time_start);end=pd.Timestamp(row.time_end)\n        a=archive[(archive.satellite==row.satellite)&(archive.acq_datetime>=start)&(archive.acq_datetime<end)]\n        if not len(a):continue\n        d=np.linalg.norm(xyz(a.latitude,a.longitude)-xyz([row.latitude],[row.longitude])[0],axis=1)\n        candidates=list(a.index[d<=chord(POLICY[\'archive_match_metres\'])]);proposals[i]=candidates\n        for j in candidates:reverse.setdefault(j,[]).append(i)\n    for i,choices in proposals.items():\n        if not choices:continue\n        if len(choices)!=1 or len(reverse[choices[0]])!=1:out.loc[i,\'archive_match_status\']=\'ambiguous\';continue\n        j=choices[0];a=archive.loc[j];out.loc[i,\'archive_match_status\']=\'unique_standard_observation\';out.loc[i,\'archive_observation_id\']=a.observation_id;out.loc[i,\'archive_type\']=int(a.type)\n        delta=np.linalg.norm(xyz([out.loc[i,\'latitude\']],[out.loc[i,\'longitude\']])-xyz([a.latitude],[a.longitude]))\n        out.loc[i,\'archive_distance_metres\']=2*RADIUS*np.arcsin(min(1,delta/2))\n    return out\n\ndef group_passes(records):\n    rows=[];f=pd.DataFrame([r for r in records if r[\'kind\']==\'fire\'])\n    for (sat,date,flag),group in f.groupby([\'satellite\',\'date\',\'day_night\'],sort=True):\n        rank=0;end=None\n        for row in group.sort_values([\'time_start\',\'id\']).to_dict(\'records\'):\n            start=pd.Timestamp(row[\'time_start\']);stop=pd.Timestamp(row[\'time_end\'])\n            if end is None or start>end+pd.Timedelta(minutes=6):rank+=1\n            pid=f\'{sat}_{date}_{flag}_{rank:02d}\'\n            rows.append({\'fire_id\':row[\'id\'],\'pass_id\':pid,\'satellite\':sat,\'date\':date,\'day_night\':flag,\'time_start\':row[\'time_start\'],\'time_end\':row[\'time_end\']})\n            end=max(end,stop) if end is not None else stop\n    return pd.DataFrame(rows)\n\ndef pair_passes(passes,window):\n    matched=[];unmatched=[]\n    for (day,flag),f in passes.groupby([\'date\',\'day_night\'],sort=True):\n        a=f[f.satellite.eq(\'Suomi-NPP\')].sort_values(\'pass_id\');b=f[f.satellite.eq(\'NOAA-20\')].sort_values(\'pass_id\')\n        good=flag in [\'DAY\',\'NIGHT\'];aids=list(a.pass_id);bids=list(b.pass_id);used=set();useda=set()\n        if good and len(a) and len(b):\n            at=np.array([pd.Timestamp(x).timestamp() for x in a.midpoint_utc]);bt=np.array([pd.Timestamp(x).timestamp() for x in b.midpoint_utc])\n            delta=np.abs(at[:,None]-bt[None,:])/60\n            eligible=(delta<=window)&a.complete_science_screen.to_numpy(bool)[:,None]&b.complete_science_screen.to_numpy(bool)[None,:]&a.local_centres.gt(0).to_numpy()[:,None]&b.local_centres.gt(0).to_numpy()[None,:]\n            cost=np.concatenate([np.where(eligible,delta,1e9),np.full((len(a),len(a)),1e6)],axis=1)\n            ai,bi=linear_sum_assignment(cost)\n            for x,y in zip(ai,bi):\n                if y>=len(b) or not eligible[x,y]:continue\n                useda.add(aids[x]);used.add(bids[y]);matched.append({\'window_minutes\':window,\'date\':day,\'day_night\':flag,\'snpp_pass\':aids[x],\'noaa20_pass\':bids[y],\'separation_minutes\':float(delta[x,y])})\n        for ids,usedids in [(aids,useda),(bids,used)]:\n            for pid in ids:\n                if pid not in usedids:unmatched.append({\'window_minutes\':window,\'pass_id\':pid,\'date\':day,\'day_night\':flag,\'status\':\'no_eligible_one_to_one_partner\'})\n    return pd.DataFrame(matched,columns=[\'window_minutes\',\'date\',\'day_night\',\'snpp_pass\',\'noaa20_pass\',\'separation_minutes\']),pd.DataFrame(unmatched)\n\ndef degree_variable(dataset,name,shape):\n    path,var=e.find_variable(dataset,name);require(var.shape==shape,\'Geometry grid mismatch: \'+name)\n    units=str(getattr(var,\'units\',\'\')).lower();require(\'degree\' in units,\'Unrecognized geometry units: \'+name)\n    return path,var\n\ndef extract_granule(fire_path,geo_path,record,region):\n    from netCDF4 import Dataset\n    native=[];support=[];bins=[];summary={\'fire_id\':record[\'id\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\'day_night\':record[\'day_night\'],\'local_centres\':0,\'clear_centres\':0,\'raw_fire_centres\':0,\'screened_fire_centres\':0,\'geometry_unavailable_centres\':0}\n    with Dataset(fire_path) as fire,Dataset(geo_path) as geo:\n        _,mask=e.find_variable(fire,\'fire mask\');_,qa=e.find_variable(fire,\'algorithm QA\');_,lat=e.find_variable(geo,\'latitude\');_,lon=e.find_variable(geo,\'longitude\')\n        require(mask.shape==qa.shape==lat.shape==lon.shape,\'Native grid mismatch\')\n        _,view=degree_variable(geo,\'sensor_zenith\',mask.shape);_,solar=degree_variable(geo,\'solar_zenith\',mask.shape)\n        # Sparse arrays are independently checked against every local fire-mask location.\n        sparse={}\n        for name in [\'FP_line\',\'FP_sample\',\'FP_latitude\',\'FP_longitude\',\'FP_confidence\',\'FP_ViewZenAng\',\'FP_SolZenAng\']:\n            _,var=e.find_variable(fire,name);sparse[name]=np.ma.asarray(var[:])\n        lengths={len(v) for v in sparse.values()};require(len(lengths)==1,\'Sparse-array length mismatch\')\n        byindex={}\n        for k,(r,c) in enumerate(zip(sparse[\'FP_line\'],sparse[\'FP_sample\'])):\n            key=(int(r),int(c));require(key not in byindex,\'Duplicate sparse fire index\');byindex[key]=k\n        seen=set()\n        for start in range(0,mask.shape[0],256):\n            sl=slice(start,min(start+256,mask.shape[0]));x=e.screen_arrays(mask[sl,:],qa[sl,:],lat[sl,:],lon[sl,:],region)\n            v=np.ma.filled(np.ma.asarray(view[sl,:]).astype(float),np.nan);s=np.ma.filled(np.ma.asarray(solar[sl,:]).astype(float),np.nan)\n            valid=np.isfinite(v)&np.isfinite(s)&(v>=0)&(v<=90)&(s>=0)&(s<=180)\n            ix=x[\'local\'];summary[\'local_centres\']+=int(ix.sum());summary[\'clear_centres\']+=int(x[\'clear\'].sum());summary[\'screened_fire_centres\']+=int(x[\'fires\'].sum());summary[\'geometry_unavailable_centres\']+=int((ix&~valid).sum())\n            raw=ix&np.isin(x[\'mask\'],[7,8,9]);summary[\'raw_fire_centres\']+=int(raw.sum())\n            for r,c in zip(*np.where(raw)):\n                key=(start+int(r),int(c));require(key in byindex,\'Fire mask location absent from sparse arrays\');k=byindex[key];seen.add(key)\n                delta=max(abs(float(sparse[\'FP_latitude\'][k])-x[\'latitude\'][r,c]),abs(float(sparse[\'FP_longitude\'][k])-x[\'longitude\'][r,c]))\n                require(np.isfinite(delta) and delta<=.01,\'Local sparse geolocation mismatch\')\n                require(int(sparse[\'FP_confidence\'][k])==int(x[\'mask\'][r,c]),\'Sparse confidence differs from mask\')\n                bits=int(x[\'qa\'][r,c]);goodqa=bool(x[\'qa_valid\'][r,c]);native.append({\'fire_id\':record[\'id\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\n                 \'line\':key[0],\'sample\':key[1],\'latitude\':float(sparse[\'FP_latitude\'][k]),\'longitude\':float(sparse[\'FP_longitude\'][k]),\'confidence_class\':int(x[\'mask\'][r,c]),\'qa\':bits,\'qa_valid\':goodqa,\'geolocation_latitude\':float(x[\'latitude\'][r,c]),\'geolocation_longitude\':float(x[\'longitude\'][r,c]),\'coordinate_difference_degrees\':float(delta),\n                 \'screen_exclusion_reason\':\',\'.join((["masked_qa"] if not goodqa else [])+(["non_nominal_input"] if goodqa and bits&127 else [])+(["residual_bowtie"] if goodqa and bits&(1<<22) else [])) or \'retained\',\n                 \'input_quality_nominal\':goodqa and (bits&127)==0,\'residual_bowtie\':bool(bits&(1<<22)) if goodqa else None,\'screened_fire\':bool(x[\'fires\'][r,c]),\n                 \'sensor_zenith_degrees\':float(v[r,c]) if valid[r,c] else np.nan,\'solar_zenith_degrees\':float(s[r,c]) if valid[r,c] else np.nan,\n                 \'sparse_view_zenith_degrees\':float(sparse[\'FP_ViewZenAng\'][k]),\'sparse_solar_zenith_degrees\':float(sparse[\'FP_SolZenAng\'][k]),\'time_basis\':\'granule_interval_not_pixel_time\'})\n            # Retain all local nominal-clear centres, including unknown geometry,\n            # in derived cache; pair screens explicitly require known geometry.\n            take=x[\'clear\'];rr,cc=np.where(take)\n            if len(rr):support.append(pd.DataFrame({\'latitude\':x[\'latitude\'][take].astype(\'float32\'),\'longitude\':x[\'longitude\'][take].astype(\'float32\'),\n              \'line\':(rr+start).astype(\'int32\'),\'sample\':cc.astype(\'int16\'),\'fire\':x[\'fires\'][take],\n              \'sensor_zenith_degrees\':np.where(valid[take],v[take],np.nan).astype(\'float32\'),\'solar_zenith_degrees\':np.where(valid[take],s[take],np.nan).astype(\'float32\')}))\n            for label,lo,hi in [(\'0_20\',0,20),(\'20_40\',20,40),(\'40_60\',40,60),(\'60_90\',60,91)]:\n                selected=ix&valid&(v>=lo)&(v<hi);bins.append({\'fire_id\':record[\'id\'],\'view_bin\':label,\'local_centres\':int(selected.sum()),\'clear_centres\':int((selected&x[\'clear\']).sum()),\'screened_fire_centres\':int((selected&x[\'fires\']).sum())})\n        # Check every sparse pixel inside the exact study geometry is represented.\n        from shapely import intersects_xy\n        inside=intersects_xy(region,np.asarray(sparse[\'FP_longitude\'],float),np.asarray(sparse[\'FP_latitude\'],float))\n        expected={(int(sparse[\'FP_line\'][k]),int(sparse[\'FP_sample\'][k])) for k in np.flatnonzero(inside)}\n        require(seen==expected,\'Local sparse/mask fire sets differ\')\n    columns=[\'fire_id\',\'satellite\',\'date\',\'time_start\',\'time_end\',\'line\',\'sample\',\'latitude\',\'longitude\',\'confidence_class\',\'qa\',\'qa_valid\',\'geolocation_latitude\',\'geolocation_longitude\',\'coordinate_difference_degrees\',\'screen_exclusion_reason\',\'input_quality_nominal\',\'residual_bowtie\',\'screened_fire\',\'sensor_zenith_degrees\',\'solar_zenith_degrees\',\'sparse_view_zenith_degrees\',\'sparse_solar_zenith_degrees\',\'time_basis\']\n    n=pd.DataFrame(native,columns=columns);p=pd.concat(support,ignore_index=True) if support else pd.DataFrame(columns=[\'latitude\',\'longitude\',\'line\',\'sample\',\'fire\',\'sensor_zenith_degrees\',\'solar_zenith_degrees\'])\n    b=pd.DataFrame(bins).groupby([\'fire_id\',\'view_bin\'],as_index=False).sum() if bins else pd.DataFrame()\n    return summary,n,p,b\n\ndef pair_diagnostics(a,b):\n    results=[]\n    for cap in POLICY[\'view_screens_degrees\']:\n        def screen(f):\n            ok=f.sensor_zenith_degrees.notna()&f.solar_zenith_degrees.notna()\n            if cap is not None:ok&=f.sensor_zenith_degrees.le(cap)\n            return f[ok].reset_index(drop=True)\n        aa=screen(a);bb=screen(b);ix,iy,d=mutual_neighbours(aa,bb)\n        for radius in [250,500]:\n            keep=d<=radius;matched=int(keep.sum());row={\'view_cap_degrees\':cap,\'distance_cap_metres\':radius,\'snpp_eligible_clear_centres\':len(aa),\'noaa20_eligible_clear_centres\':len(bb),\n             \'mutual_nearest_clear_centre_pairs\':matched,\'snpp_fire_centres_on_support\':int(aa.iloc[ix[keep]].fire.sum()) if matched else None,\n             \'noaa20_fire_centres_on_support\':int(bb.iloc[iy[keep]].fire.sum()) if matched else None,\n             \'median_separation_metres\':float(np.median(d[keep])) if matched else None,\'status\':\'point_centre_diagnostic_not_physical_footprint_match\'}\n            results.append(row)\n    return pd.DataFrame(results)\n\ndef cached_science_files(inv,receipts,cache):\n    cache=Path(cache);rmap={r[\'id\']:r for r in receipts};paths={}\n    for index,r in enumerate(inv[\'records\'],1):\n        require(r[\'id\'] in rmap,\'Missing expected science receipt\');receipt=rmap[r[\'id\']];name=Path(urllib.parse.urlsplit(r[\'download_url\']).path).name\n        require(name==r[\'name\']+\'.nc\' or name==r[\'name\'],\'Cache filename mismatch\');path=cache/name\n        require(path.exists(),\'Missing cached file: \'+name+\'; locate the original science_cache, no download attempted\')\n        require(path.stat().st_size==receipt[\'bytes\'] and file_sha(path)==receipt[\'sha256\'],\'Cached science file differs: \'+name)\n        paths[r[\'id\']]=path\n        if index%8==0 or index==len(inv[\'records\']):print(\'Verified cached science files\',index,\'/\',len(inv[\'records\']),flush=True)\n    return paths\n\ndef run(inv,receipts,boundary,archive,previous_granules,cache,output,derived,method_hash):\n    from shapely.geometry import shape\n    require(e.sha(boundary)==e.BOUNDARY_SHA,\'Boundary changed\');region=shape(json.loads(boundary)[\'features\'][0][\'geometry\'])\n    paths=cached_science_files(inv,receipts,cache);output=Path(output);output.mkdir(parents=True,exist_ok=True);derived=Path(derived);derived.mkdir(parents=True,exist_ok=True)\n    lookup={r[\'id\']:r for r in inv[\'records\']};rmap={r[\'id\']:r for r in receipts};summaries=[];native=[];bins=[];failures=[];support_paths={}\n    for join in inv[\'companions\']:\n        f=lookup[join[\'fire_id\']];g=lookup[join[\'geo_id\']];key=e.sha(f[\'id\'].encode())[:20];folder=derived/key;folder.mkdir(exist_ok=True)\n        signature={\'method_sha256\':method_hash,\'fire_sha256\':rmap[f[\'id\']][\'sha256\'],\'geo_sha256\':rmap[g[\'id\']][\'sha256\']};checkpoint=folder/\'checkpoint.json\'\n        try:\n            if checkpoint.exists():\n                old=json.loads(checkpoint.read_text());require(old[\'signature\']==signature,\'Derived cache method/input changed; use a fresh derived folder\')\n                for name,h in old[\'hashes\'].items():require(file_sha(folder/name)==h,\'Derived cache changed\')\n                summary=json.loads((folder/\'summary.json\').read_text());n=pd.read_parquet(folder/\'native.parquet\');b=pd.read_parquet(folder/\'bins.parquet\')\n            else:\n                summary,n,p,b=extract_granule(paths[f[\'id\']],paths[g[\'id\']],f,region)\n                n.to_parquet(folder/\'native.parquet\',index=False);p.to_parquet(folder/\'clear_centres.parquet\',index=False);b.to_parquet(folder/\'bins.parquet\',index=False);(folder/\'summary.json\').write_text(json.dumps(summary,indent=2)+\'\\n\')\n                checkpoint.write_text(json.dumps({\'signature\':signature,\'hashes\':{name:file_sha(folder/name) for name in [\'native.parquet\',\'clear_centres.parquet\',\'bins.parquet\',\'summary.json\']}},indent=2)+\'\\n\')\n            previous=previous_granules[previous_granules.fire_id.eq(f[\'id\'])].iloc[0]\n            require(summary[\'local_centres\']==previous.local_geolocated_centres and summary[\'clear_centres\']==previous.nominal_clear_centres and summary[\'screened_fire_centres\']==previous.screened_fire_centres,\'Reprocessed screening differs from completed pilot\')\n            require(summary[\'raw_fire_centres\']==previous[[\'class_7\',\'class_8\',\'class_9\']].sum(),\'Reprocessed native count differs\')\n            summaries.append(summary);native.append(n);bins.append(b);support_paths[f[\'id\']]=folder/\'clear_centres.parquet\';print(\'Reprocessed\',f[\'name\'],flush=True)\n        except Exception as error:\n            failures.append({\'fire_id\':f[\'id\'],\'error_type\':type(error).__name__,\'reason\':str(error)[:240]});print(\'Unavailable cached audit:\',f[\'name\'],type(error).__name__,flush=True)\n        (output/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    members=group_passes(inv[\'records\']);summary=pd.DataFrame(summaries);pass_rows=[]\n    for pid,group in members.groupby(\'pass_id\',sort=True):\n        local=summary[summary.fire_id.isin(group.fire_id)] if len(summary) else pd.DataFrame();start=min(pd.Timestamp(x) for x in group.time_start);end=max(pd.Timestamp(x) for x in group.time_end)\n        pass_rows.append({\'pass_id\':pid,\'satellite\':group.iloc[0].satellite,\'date\':group.iloc[0].date,\'day_night\':group.iloc[0].day_night,\'time_start\':start.isoformat(),\'time_end\':end.isoformat(),\'midpoint_utc\':(start+(end-start)/2).isoformat(),\n         \'granules\':len(group),\'complete_science_screen\':len(local)==len(group),\'local_centres\':int(local.local_centres.sum()) if len(local)==len(group) else None,\'nominal_clear_centres\':int(local.clear_centres.sum()) if len(local)==len(group) and local.local_centres.sum()>0 else None,\'native_fire_centres\':int(local.raw_fire_centres.sum()) if len(local)==len(group) and local.local_centres.sum()>0 else None,\'screened_fire_centres\':int(local.screened_fire_centres.sum()) if len(local)==len(group) and local.local_centres.sum()>0 else None})\n    passes=pd.DataFrame(pass_rows);matched=[];unmatched=[];diagnostics=[];done_pairs={}\n    def pass_support(pid):\n        files=[support_paths[i] for i in members[members.pass_id.eq(pid)].fire_id if i in support_paths]\n        return pd.concat([pd.read_parquet(p) for p in files],ignore_index=True) if files else pd.DataFrame(columns=[\'latitude\',\'longitude\',\'fire\',\'sensor_zenith_degrees\',\'solar_zenith_degrees\'])\n    for window in [30,60,90]:\n        pairs,un=pair_passes(passes,window);matched.append(pairs);unmatched.append(un)\n        for row in pairs.to_dict(\'records\'):\n            key=(row[\'snpp_pass\'],row[\'noaa20_pass\'])\n            if key not in done_pairs:done_pairs[key]=pair_diagnostics(pass_support(key[0]),pass_support(key[1]));print(\'Compared pass centres\',key,flush=True)\n            d=done_pairs[key].copy()\n            for name,value in row.items():d[name]=value\n            diagnostics.append(d)\n    native_frame=pd.concat(native,ignore_index=True) if native else pd.DataFrame(columns=[\'fire_id\',\'satellite\',\'latitude\',\'longitude\',\'time_start\',\'time_end\'])\n    typed=archive_match(native_frame,archive).merge(members[[\'fire_id\',\'pass_id\']],on=\'fire_id\',how=\'left\',validate=\'many_to_one\');frames={\'native_fire_pixel_audit\':typed,\'granule_audit\':summary,\'view_geometry_bins\':pd.concat(bins,ignore_index=True) if bins else pd.DataFrame(),\n     \'pass_membership\':members,\'pass_inventory\':passes,\'matched_passes\':pd.concat(matched,ignore_index=True),\'unmatched_passes\':pd.concat(unmatched,ignore_index=True),\n     \'mutual_centre_diagnostics\':pd.concat(diagnostics,ignore_index=True) if diagnostics else pd.DataFrame()}\n    for name,frame in frames.items():frame.to_parquet(output/(name+\'.parquet\'),index=False)\n    unmatched_archive=archive[~archive.observation_id.isin(typed.archive_observation_id.dropna())];unmatched_archive.to_parquet(output/\'unmatched_standard_observations.parquet\',index=False)\n    report={\'audit_id\':POLICY[\'audit_id\'],\'input_science_files\':len(paths),\'granules_reprocessed\':len(summary),\'granules_failed\':len(failures),\'native_regional_fire_pixels\':len(typed),\n      \'unique_archive_matches\':int(typed.archive_match_status.eq(\'unique_standard_observation\').sum()),\'ambiguous_archive_matches\':int(typed.archive_match_status.eq(\'ambiguous\').sum()),\'unmatched_standard_observations\':len(unmatched_archive),\n      \'status\':\'exploratory_pass_and_point_audit_only\',\'coverage_verified\':False,\'replacement_validated\':False,\'uncertainty_validated\':False,\'frontend_changed\':False,\'network_downloads\':0,\n      \'limitations\':[\'Only six purposively sampled dates\',\'Point-centre proximity is not physical footprint coverage\',\'Granule intervals, not exact pixel observation times\',\'Type attribution carried from standard archive, not independent fire truth\',\'Window/view/distance sensitivities are not independent validation samples\']}\n    (output/\'decision.json\').write_text(json.dumps(report,indent=2)+\'\\n\');(output/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    (output/\'manifest.json\').write_text(json.dumps({\'decision\':report,\'files\':{p.name:{\'bytes\':p.stat().st_size,\'sha256\':file_sha(p)} for p in sorted(output.iterdir()) if p.is_file() and p.name!=\'manifest.json\'}},indent=2)+\'\\n\')\n    return report\n'
base_namespace={'__file__':str((OUTPUT_ROOT/'pipeline'/'base.py').resolve()),'__name__':'cached_base'}
exec(compile(BASE_METHOD_SOURCE,'frozen_evidence_base.py','exec'),base_namespace)
audit_namespace={'__file__':str((OUTPUT_ROOT/'pipeline'/'audit.py').resolve()),'__name__':'cached_audit','e':types.SimpleNamespace(**base_namespace)}
exec(compile(AUDIT_METHOD_SOURCE.replace('from . import atlas_observation_evidence as e',''),'frozen_cached_pixel_audit.py','exec'),audit_namespace)
METHOD_HASH=hashlib.sha256((BASE_METHOD_SOURCE+AUDIT_METHOD_SOURCE+hashlib.sha256(INPUT_BYTES).hexdigest()).encode()).hexdigest()
DERIVED=OUTPUT_ROOT/('derived_'+METHOD_HASH[:12]);DERIVED.mkdir(exist_ok=True)
(RESULTS/'base_implementation.py').write_text(BASE_METHOD_SOURCE)
(RESULTS/'audit_implementation.py').write_text(AUDIT_METHOD_SOURCE)


21577

## 5. Reprocess and compare the cached files
The initial SHA256 check reads approximately 5 GiB from Drive; it does not download new NASA files. Reading cached files may take several minutes. Each completed granule has a derived checkpoint; reruns reuse it after checking its hashes. The derived clear-centre cache may need several hundred additional MiB. It is excluded from the final review ZIP.

Reprocessed totals must reproduce your completed pilot before being included. Any failed audit stays unavailable. Geometry changes, unknown units, missing sparse pixels or ambiguous archive matches must not be silently repaired into valid observations.

In [4]:
DECISION=audit_namespace['run'](
    INPUTS['inventory'],INPUTS['receipts'],INPUTS['boundary'].encode(),ARCHIVE,PREVIOUS,
    CACHE,RESULTS,DERIVED,METHOD_HASH)
print(json.dumps(DECISION,indent=2))


Verified cached science files 8 / 68
Verified cached science files 16 / 68
Verified cached science files 24 / 68
Verified cached science files 32 / 68
Verified cached science files 40 / 68
Verified cached science files 48 / 68
Verified cached science files 56 / 68
Verified cached science files 64 / 68
Verified cached science files 68 / 68
Reprocessed VNP14IMG.A2021227.0830.002.2024074081032
Reprocessed VNP14IMG.A2021227.1924.002.2024074081032
Reprocessed VNP14IMG.A2021227.1930.002.2024074081029
Reprocessed VNP14IMG.A2021227.2106.002.2024074081029
Reprocessed VJ114IMG.A2021227.0742.002.2024081173830
Reprocessed VJ114IMG.A2021227.2018.002.2024081173829
Reprocessed VNP14IMG.A2021231.0718.002.2024074084526
Reprocessed VNP14IMG.A2021231.0854.002.2024074084527
Reprocessed VNP14IMG.A2021231.1954.002.2024074084534
Reprocessed VJ114IMG.A2021231.0806.002.2024081184907
Reprocessed VJ114IMG.A2021231.1900.002.2024081184905
Reprocessed VJ114IMG.A2021231.2042.002.2024081184906
Reprocessed VJ114IMG.A2

## 6. Inspect the audit
Review failures first. Native counts, stricter quality-screened counts and archive type attribution are separate fields. Each pass is used once per temporal-window sensitivity; passes can reappear across sensitivities. Zero matched centres means no shared support for that diagnostic, so fire counts on that support remain unavailable. No replacement accuracy or uncertainty score is claimed from this six-day feasibility sample.

In [5]:
from IPython.display import display
display(pd.DataFrame(json.loads((RESULTS/'failures.json').read_text())))
display(pd.read_parquet(RESULTS/'native_fire_pixel_audit.parquet'))
display(pd.read_parquet(RESULTS/'pass_inventory.parquet'))
display(pd.read_parquet(RESULTS/'matched_passes.parquet'))
display(pd.read_parquet(RESULTS/'unmatched_passes.parquet'))
display(pd.read_parquet(RESULTS/'mutual_centre_diagnostics.parquet'))


""


,fire_id,satellite,date,time_start,time_end,line,sample,latitude,longitude,confidence_class,...,sensor_zenith_degrees,solar_zenith_degrees,sparse_view_zenith_degrees,sparse_solar_zenith_degrees,time_basis,archive_match_status,archive_observation_id,archive_type,archive_distance_metres,pass_id
0,G2919341305-LPCLOUD,Suomi-NPP,2021-08-19,2021-08-19T07:18:00.000Z,2021-08-19T07:24:00.000Z,375,5491,6.322983,81.356331,8,...,57.349998,11.759999,57.349998,11.759999,granule_interval_not_pixel_time,unique_standard_observation,7a60341a87a82679ac7b4f133407087ddb053b695b9f21...,0,0.324641,Suomi-NPP_2021-08-19_DAY_01
1,G2919341305-LPCLOUD,Suomi-NPP,2021-08-19,2021-08-19T07:18:00.000Z,2021-08-19T07:24:00.000Z,377,5497,6.330317,81.333534,8,...,57.419998,11.730000,57.419998,11.730000,granule_interval_not_pixel_time,unique_standard_observation,6e5e3d52c0bf2963cf3300879d79b043006c084dd0f49d...,0,0.573954,Suomi-NPP_2021-08-19_DAY_01
2,G2919341305-LPCLOUD,Suomi-NPP,2021-08-19,2021-08-19T07:18:00.000Z,2021-08-19T07:24:00.000Z,377,5498,6.329712,81.329987,8,...,57.430000,11.730000,57.430000,11.730000,granule_interval_not_pixel_time,unique_standard_observation,727acf639f4aeb6f7e21cdf0db0411f37f25d7834f1f53...,0,0.462803,Suomi-NPP_2021-08-19_DAY_01
3,G2919341305-LPCLOUD,Suomi-NPP,2021-08-19,2021-08-19T07:18:00.000Z,2021-08-19T07:24:00.000Z,388,5489,6.323977,81.355316,9,...,57.320000,11.759999,57.320000,11.759999,granule_interval_not_pixel_time,unique_standard_observation,e906b9f305670b32d899b64ad0464dd52bc60b6206fa98...,0,0.574394,Suomi-NPP_2021-08-19_DAY_01
4,G2919341305-LPCLOUD,Suomi-NPP,2021-08-19,2021-08-19T07:18:00.000Z,2021-08-19T07:24:00.000Z,389,5495,6.325685,81.333405,9,...,57.399998,11.740000,57.399998,11.740000,granule_interval_not_pixel_time,unique_standard_observation,52b8cf32e06ad679ae8959287a5283f9aa63c465958d1c...,0,0.712426,Suomi-NPP_2021-08-19_DAY_01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,G2921978602-LPCLOUD,NOAA-20,2022-08-12,2022-08-12T07:54:00.000Z,2022-08-12T08:00:00.000Z,1569,3123,7.271233,81.560074,8,...,2.430000,20.279999,2.430000,20.279999,granule_interval_not_pixel_time,unique_standard_observation,3359f90449f4e2ec19c9dfd6550406c714b1d48d4beddb...,0,0.514272,NOAA-20_2022-08-12_DAY_01
331,G2921978602-LPCLOUD,NOAA-20,2022-08-12,2022-08-12T07:54:00.000Z,2022-08-12T08:00:00.000Z,1580,3221,7.256386,81.215729,8,...,0.550000,19.969999,0.550000,19.969999,granule_interval_not_pixel_time,unique_standard_observation,8cae8f6f93f18e47275aa910b8ea38fc5b186fb14dbb3c...,0,0.435675,NOAA-20_2022-08-12_DAY_01
332,G2921978602-LPCLOUD,NOAA-20,2022-08-12,2022-08-12T07:54:00.000Z,2022-08-12T08:00:00.000Z,1881,3510,8.090590,79.985558,8,...,9.270000,18.580000,9.270000,18.580000,granule_interval_not_pixel_time,unique_standard_observation,dfd97ff84ace251052f67079d1e9035c968da6e81291c2...,0,0.274217,NOAA-20_2022-08-12_DAY_01
333,G2921975979-LPCLOUD,NOAA-20,2022-08-12,2022-08-12T20:30:00.000Z,2022-08-12T20:36:00.000Z,2547,4922,6.630124,81.188927,8,...,47.939999,144.919998,47.939999,144.919998,granule_interval_not_pixel_time,unique_standard_observation,c09733854ed6398809e45e5f26f082e5d27782ea72891a...,0,0.542878,NOAA-20_2022-08-12_NIGHT_01


,pass_id,satellite,date,day_night,time_start,time_end,midpoint_utc,granules,complete_science_screen,local_centres,nominal_clear_centres,native_fire_centres,screened_fire_centres
0,NOAA-20_2021-08-15_DAY_01,NOAA-20,2021-08-15,DAY,2021-08-15T07:42:00+00:00,2021-08-15T07:48:00+00:00,2021-08-15T07:45:00+00:00,1,True,373359,6605.0,0.0,0.0
1,NOAA-20_2021-08-15_NIGHT_01,NOAA-20,2021-08-15,NIGHT,2021-08-15T20:18:00+00:00,2021-08-15T20:24:00+00:00,2021-08-15T20:21:00+00:00,1,True,368439,72932.0,0.0,0.0
2,NOAA-20_2021-08-19_DAY_01,NOAA-20,2021-08-19,DAY,2021-08-19T08:06:00+00:00,2021-08-19T08:12:00+00:00,2021-08-19T08:09:00+00:00,1,True,390294,330438.0,220.0,202.0
3,NOAA-20_2021-08-19_NIGHT_01,NOAA-20,2021-08-19,NIGHT,2021-08-19T19:00:00+00:00,2021-08-19T19:06:00+00:00,2021-08-19T19:03:00+00:00,1,True,0,NaN,NaN,NaN
4,NOAA-20_2021-08-19_NIGHT_02,NOAA-20,2021-08-19,NIGHT,2021-08-19T20:42:00+00:00,2021-08-19T20:48:00+00:00,2021-08-19T20:45:00+00:00,1,True,436843,139246.0,1.0,1.0
5,NOAA-20_2022-08-04_DAY_01,NOAA-20,2022-08-04,DAY,2022-08-04T07:00:00+00:00,2022-08-04T07:12:00+00:00,2022-08-04T07:06:00+00:00,2,True,248218,1967.0,0.0,0.0
6,NOAA-20_2022-08-04_DAY_02,NOAA-20,2022-08-04,DAY,2022-08-04T08:42:00+00:00,2022-08-04T08:48:00+00:00,2022-08-04T08:45:00+00:00,1,True,299791,2211.0,0.0,0.0
7,NOAA-20_2022-08-04_NIGHT_01,NOAA-20,2022-08-04,NIGHT,2022-08-04T19:36:00+00:00,2022-08-04T19:48:00+00:00,2022-08-04T19:42:00+00:00,2,True,377421,50249.0,0.0,0.0
8,NOAA-20_2022-08-12_DAY_01,NOAA-20,2022-08-12,DAY,2022-08-12T07:54:00+00:00,2022-08-12T08:00:00+00:00,2022-08-12T07:57:00+00:00,1,True,456675,167804.0,12.0,11.0
9,NOAA-20_2022-08-12_NIGHT_01,NOAA-20,2022-08-12,NIGHT,2022-08-12T20:30:00+00:00,2022-08-12T20:36:00+00:00,2022-08-12T20:33:00+00:00,1,True,373345,212302.0,1.0,1.0


,window_minutes,date,day_night,snpp_pass,noaa20_pass,separation_minutes
0,30,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
1,30,2024-06-05,NIGHT,Suomi-NPP_2024-06-05_NIGHT_01,NOAA-20_2024-06-05_NIGHT_01,27.0
2,60,2021-08-15,DAY,Suomi-NPP_2021-08-15_DAY_01,NOAA-20_2021-08-15_DAY_01,48.0
3,60,2021-08-15,NIGHT,Suomi-NPP_2021-08-15_NIGHT_01,NOAA-20_2021-08-15_NIGHT_01,51.0
4,60,2021-08-19,DAY,Suomi-NPP_2021-08-19_DAY_01,NOAA-20_2021-08-19_DAY_01,48.0
5,60,2021-08-19,NIGHT,Suomi-NPP_2021-08-19_NIGHT_01,NOAA-20_2021-08-19_NIGHT_02,48.0
6,60,2022-08-12,DAY,Suomi-NPP_2022-08-12_DAY_02,NOAA-20_2022-08-12_DAY_01,48.0
7,60,2022-08-12,NIGHT,Suomi-NPP_2022-08-12_NIGHT_01,NOAA-20_2022-08-12_NIGHT_01,48.0
8,60,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
9,60,2024-06-05,NIGHT,Suomi-NPP_2024-06-05_NIGHT_01,NOAA-20_2024-06-05_NIGHT_01,27.0


,window_minutes,pass_id,date,day_night,status
0,30,Suomi-NPP_2021-08-15_DAY_01,2021-08-15,DAY,no_eligible_one_to_one_partner
1,30,NOAA-20_2021-08-15_DAY_01,2021-08-15,DAY,no_eligible_one_to_one_partner
2,30,Suomi-NPP_2021-08-15_NIGHT_01,2021-08-15,NIGHT,no_eligible_one_to_one_partner
3,30,Suomi-NPP_2021-08-15_NIGHT_02,2021-08-15,NIGHT,no_eligible_one_to_one_partner
4,30,NOAA-20_2021-08-15_NIGHT_01,2021-08-15,NIGHT,no_eligible_one_to_one_partner
5,30,Suomi-NPP_2021-08-19_DAY_01,2021-08-19,DAY,no_eligible_one_to_one_partner
6,30,Suomi-NPP_2021-08-19_DAY_02,2021-08-19,DAY,no_eligible_one_to_one_partner
7,30,NOAA-20_2021-08-19_DAY_01,2021-08-19,DAY,no_eligible_one_to_one_partner
8,30,Suomi-NPP_2021-08-19_NIGHT_01,2021-08-19,NIGHT,no_eligible_one_to_one_partner
9,30,NOAA-20_2021-08-19_NIGHT_01,2021-08-19,NIGHT,no_eligible_one_to_one_partner


,view_cap_degrees,distance_cap_metres,snpp_eligible_clear_centres,noaa20_eligible_clear_centres,mutual_nearest_clear_centre_pairs,snpp_fire_centres_on_support,noaa20_fire_centres_on_support,median_separation_metres,status,window_minutes,date,day_night,snpp_pass,noaa20_pass,separation_minutes
0,NaN,250,140726,90180,53878,0.0,0.0,154.709416,point_centre_diagnostic_not_physical_footprint...,30,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
1,NaN,500,140726,90180,59497,0.0,0.0,163.910496,point_centre_diagnostic_not_physical_footprint...,30,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
2,40.0,250,140726,0,0,NaN,NaN,NaN,point_centre_diagnostic_not_physical_footprint...,30,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
3,40.0,500,140726,0,0,NaN,NaN,NaN,point_centre_diagnostic_not_physical_footprint...,30,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
4,60.0,250,140726,18878,12264,0.0,0.0,151.525898,point_centre_diagnostic_not_physical_footprint...,30,2024-06-05,DAY,Suomi-NPP_2024-06-05_DAY_01,NOAA-20_2024-06-05_DAY_02,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,NaN,500,3440,82688,1826,0.0,0.0,193.658850,point_centre_diagnostic_not_physical_footprint...,90,2024-06-05,NIGHT,Suomi-NPP_2024-06-05_NIGHT_01,NOAA-20_2024-06-05_NIGHT_01,27.0
104,40.0,250,0,0,0,NaN,NaN,NaN,point_centre_diagnostic_not_physical_footprint...,90,2024-06-05,NIGHT,Suomi-NPP_2024-06-05_NIGHT_01,NOAA-20_2024-06-05_NIGHT_01,27.0
105,40.0,500,0,0,0,NaN,NaN,NaN,point_centre_diagnostic_not_physical_footprint...,90,2024-06-05,NIGHT,Suomi-NPP_2024-06-05_NIGHT_01,NOAA-20_2024-06-05_NIGHT_01,27.0
106,60.0,250,0,82688,0,NaN,NaN,NaN,point_centre_diagnostic_not_physical_footprint...,90,2024-06-05,NIGHT,Suomi-NPP_2024-06-05_NIGHT_01,NOAA-20_2024-06-05_NIGHT_01,27.0


## 7. Download one review ZIP
Save this notebook with its outputs and add the ZIP to the project folder. It contains input provenance, pixel audit, pass membership and diagnostics, failures, method, runtime versions and hashes. It excludes both the original science cache and derived clear-centre cache. Keep those caches for reuse.

After this review we can freeze a seasonally balanced multi-year simulated-gap study with separate fit/calibration/evaluation periods. Six purposive days cannot validate replacement uncertainty, early historical bands, annual intervals or district allocation.

In [6]:
from datetime import datetime,timezone
import importlib.metadata
versions={name:importlib.metadata.version(name) for name in ['numpy','pandas','pyarrow','shapely','h3','scipy','netCDF4']}
(OUTPUT_ROOT/'runtime_versions.json').write_text(json.dumps(versions,indent=2)+'\n')
archive_path=OUTPUT_ROOT/('cindertrace_atlas_lk_pixel_pass_audit_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip')
members={'verified_inputs.json':(OUTPUT_ROOT/'verified_inputs.json').read_bytes(),'runtime_versions.json':(OUTPUT_ROOT/'runtime_versions.json').read_bytes()}
for path in sorted(RESULTS.iterdir()):
    if path.is_file():members['results/'+path.name]=path.read_bytes()
manifest={name:{'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()} for name,raw in sorted(members.items())}
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for name,raw in sorted(members.items()):archive.writestr(name,raw)
    archive.writestr('bundle_manifest.json',json.dumps(manifest,indent=2)+'\n')
print('Saved:',archive_path.resolve())
print('SHA256:',hashlib.sha256(archive_path.read_bytes()).hexdigest())
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))


Saved: /content/drive/MyDrive/CinderTrace_Observation_Evidence_v1/pixel_pass_audit_v1/cindertrace_atlas_lk_pixel_pass_audit_20261115T004550Z.zip
SHA256: dea71170261cd155b5ff7515b12b3078bba08b280cd2204c48dd23ca5ac46de3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>